# Heart Disease Prediction System
**Machine Learning Classification Model**

This notebook implements a complete ML pipeline to predict the presence of heart disease using the Cleveland Heart Disease dataset from the UCI Machine Learning Repository.

**Pipeline:**
1. Data Loading & Exploration
2. Data Cleaning & Preprocessing
3. Exploratory Data Analysis (EDA)
4. Feature Selection
5. Model Training & Evaluation
6. Model Optimization
7. Final Results

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectKBest, chi2, f_classif, RFE
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_score, recall_score, f1_score
)
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('Libraries loaded successfully.')

## 2. Load Dataset

The **Cleveland Heart Disease Dataset** from the UCI ML Repository contains 303 patient records with 13 clinical features and a target label indicating heart disease presence.

In [ ]:
# Load the Cleveland Heart Disease dataset directly from UCI ML Repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'

column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

df = pd.read_csv(url, header=None, names=column_names, na_values='?')

# The original target has values 0-4; binarize to 0 (no disease) vs 1 (disease)
df['target'] = (df['target'] > 0).astype(int)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 3. Data Exploration

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Descriptive Statistics ===')
df.describe()

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0])

print('\n=== Target Distribution ===')
print(df['target'].value_counts())
print(f'No Heart Disease: {(df["target"]==0).sum()} ({(df["target"]==0).mean()*100:.1f}%)')
print(f'Heart Disease:    {(df["target"]==1).sum()} ({(df["target"]==1).mean()*100:.1f}%)')

## 4. Data Cleaning & Preprocessing

In [ ]:
# Handle missing values — 'ca' and 'thal' have a small number of NaN rows
print(f'Rows before dropping NaN: {len(df)}')
df.dropna(inplace=True)
print(f'Rows after dropping NaN:  {len(df)}')

# Convert float columns that should be int (artifact of loading with NaN)
int_cols = ['ca', 'thal']
df[int_cols] = df[int_cols].astype(int)

# Verify dtypes
print('\n=== Updated dtypes ===')
print(df.dtypes)

In [ ]:
# Check for duplicate rows
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}')
if dupes > 0:
    df.drop_duplicates(inplace=True)
    print(f'Removed duplicates. New shape: {df.shape}')

In [ ]:
# Outlier detection using IQR method on continuous features
continuous_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(1, len(continuous_features), figsize=(18, 5))
for i, col in enumerate(continuous_features):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)
    axes[i].set_xlabel('')

plt.suptitle('Boxplots for Outlier Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Report outliers per feature
for col in continuous_features:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f'{col}: {len(outliers)} outliers')

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Target class distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

counts = df['target'].value_counts()
ax1.bar(['No Disease', 'Heart Disease'], counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
ax1.set_title('Target Class Distribution', fontweight='bold')
ax1.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax1.text(i, v + 1, str(v), ha='center', fontweight='bold')

ax2.pie(counts.values, labels=['No Disease', 'Heart Disease'], autopct='%1.1f%%',
        colors=['#2ecc71', '#e74c3c'], startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax2.set_title('Target Class Proportion', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of continuous features by target class
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(continuous_features):
    for target_val, label, color in [(0, 'No Disease', '#2ecc71'), (1, 'Heart Disease', '#e74c3c')]:
        axes[i].hist(df[df['target'] == target_val][col], alpha=0.6, label=label, color=color, bins=20)
    axes[i].set_title(f'{col} Distribution', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].legend()

axes[-1].set_visible(False)
plt.suptitle('Feature Distributions by Target Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features vs target
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
cat_labels = {
    'sex': {0: 'Female', 1: 'Male'},
    'cp': {0: 'Typical\nAngina', 1: 'Atypical\nAngina', 2: 'Non-Anginal', 3: 'Asymptomatic'},
    'fbs': {0: 'Normal', 1: '>120 mg/dl'},
    'exang': {0: 'No', 1: 'Yes'},
}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_features):
    ct = pd.crosstab(df[col], df['target'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'], edgecolor='white', rot=0)
    axes[i].set_title(f'{col} vs Heart Disease', fontweight='bold')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].legend(['No Disease', 'Heart Disease'], fontsize=9)

plt.suptitle('Categorical Features vs Target Class (%)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(13, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    linewidths=0.5, square=True, vmin=-1, vmax=1,
    annot_kws={'size': 10}
)
plt.title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Top correlations with target
print('=== Feature Correlations with Target ===')
target_corr = df.corr()['target'].drop('target').sort_values(key=abs, ascending=False)
print(target_corr.to_string())

## 6. Feature Selection

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

feature_names = X.columns.tolist()
print(f'Features: {feature_names}')
print(f'Total samples: {len(X)}, Positive (disease): {y.sum()}, Negative: {(y==0).sum()}')

In [ ]:
# --- Method 1: SelectKBest (ANOVA F-test) ---
scaler_temp = StandardScaler()
X_scaled_temp = scaler_temp.fit_transform(X)

selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X_scaled_temp, y)

f_scores = pd.DataFrame({
    'Feature': feature_names,
    'F_Score': selector.scores_,
    'P_Value': selector.pvalues_
}).sort_values('F_Score', ascending=False)

print('=== ANOVA F-Test Feature Scores ===')
print(f_scores.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in f_scores['P_Value']]
ax.barh(f_scores['Feature'], f_scores['F_Score'], color=colors)
ax.set_xlabel('F-Score')
ax.set_title('Feature Importance — ANOVA F-Test\n(Red = statistically significant, p < 0.05)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# --- Method 2: Random Forest Feature Importance ---
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_scaled_temp, y)

rf_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print('=== Random Forest Feature Importance ===')
print(rf_importance.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(rf_importance['Feature'], rf_importance['Importance'], color='#3498db')
ax.set_xlabel('Importance Score')
ax.set_title('Feature Importance — Random Forest', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# --- Method 3: Recursive Feature Elimination (RFE) with Logistic Regression ---
lr_rfe = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=lr_rfe, n_features_to_select=10)
rfe.fit(X_scaled_temp, y)

rfe_results = pd.DataFrame({
    'Feature': feature_names,
    'Selected': rfe.support_,
    'Rank': rfe.ranking_
}).sort_values('Rank')

print('=== RFE Feature Selection ===')
print(rfe_results.to_string(index=False))

selected_features = rfe_results[rfe_results['Selected']]['Feature'].tolist()
print(f'\nSelected features ({len(selected_features)}): {selected_features}')

In [ ]:
# Use all 13 features — all contribute meaningfully
# (RFE selecting 10 vs all 13 shows minimal accuracy difference)
X_final = X.copy()
print(f'Final feature set: {X_final.columns.tolist()}')
print(f'Final shape: {X_final.shape}')

## 7. Train/Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Train positive rate: {y_train.mean():.3f}')
print(f'Test positive rate:  {y_test.mean():.3f}')

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('\nStandardization complete. Train mean ~ 0, std ~ 1.')

## 8. Model Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
    'SVM':                 SVC(probability=True, random_state=42),
    'KNN':                 KNeighborsClassifier(),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    results[name] = cv_scores
    print(f'{name:<25} CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# Plot CV results
fig, ax = plt.subplots(figsize=(12, 6))
model_names = list(results.keys())
means = [results[m].mean() for m in model_names]
stds  = [results[m].std()  for m in model_names]

bars = ax.barh(model_names, means, xerr=stds, color='#3498db', alpha=0.8, capsize=5, edgecolor='white')
ax.set_xlabel('5-Fold Cross-Validation Accuracy')
ax.set_title('Model Comparison — 5-Fold Cross-Validation', fontweight='bold')
ax.set_xlim(0.6, 1.0)
for bar, mean in zip(bars, means):
    ax.text(mean + 0.003, bar.get_y() + bar.get_height()/2,
            f'{mean:.3f}', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Train & Evaluate Final Model (Logistic Regression)

In [ ]:
# Train the final Logistic Regression model
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)[:, 1]

acc       = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_prob)

print('=== Logistic Regression — Test Set Performance ===')
print(f'  Accuracy:  {acc:.4f} ({acc*100:.2f}%)')
print(f'  Precision: {precision:.4f}')
print(f'  Recall:    {recall:.4f}')
print(f'  F1 Score:  {f1:.4f}')
print(f'  ROC-AUC:   {roc_auc:.4f}')
print()
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Heart Disease']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['No Disease', 'Heart Disease'],
            yticklabels=['No Disease', 'Heart Disease'],
            linewidths=1, linecolor='white', annot_kws={'size': 14, 'weight': 'bold'})
ax1.set_xlabel('Predicted Label', fontsize=12)
ax1.set_ylabel('True Label', fontsize=12)
ax1.set_title('Confusion Matrix', fontweight='bold', fontsize=13)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax2.fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve', fontweight='bold', fontsize=13)
ax2.legend(loc='lower right')
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1.02])

plt.tight_layout()
plt.show()

## 10. Model Optimization — Hyperparameter Tuning

In [ ]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2', 'elasticnet'],
    'solver': ['saga'],
    'l1_ratio': [0.1, 0.5, 0.9]  # only used with elasticnet
}

grid_search = GridSearchCV(
    LogisticRegression(max_iter=5000, random_state=42),
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train_scaled, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV accuracy: {grid_search.best_score_:.4f}')

best_model = grid_search.best_estimator_

In [ ]:
# Evaluate optimized model on test set
y_pred_opt = best_model.predict(X_test_scaled)
y_prob_opt = best_model.predict_proba(X_test_scaled)[:, 1]

acc_opt       = accuracy_score(y_test, y_pred_opt)
precision_opt = precision_score(y_test, y_pred_opt)
recall_opt    = recall_score(y_test, y_pred_opt)
f1_opt        = f1_score(y_test, y_pred_opt)
roc_auc_opt   = roc_auc_score(y_test, y_prob_opt)

print('=== Optimized Logistic Regression — Test Set Performance ===')
print(f'  Accuracy:  {acc_opt:.4f} ({acc_opt*100:.2f}%)')
print(f'  Precision: {precision_opt:.4f}')
print(f'  Recall:    {recall_opt:.4f}')
print(f'  F1 Score:  {f1_opt:.4f}')
print(f'  ROC-AUC:   {roc_auc_opt:.4f}')
print()
print('=== Classification Report ===')
print(classification_report(y_test, y_pred_opt, target_names=['No Disease', 'Heart Disease']))

## 11. Feature Coefficients (Model Interpretability)

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X_final.columns,
    'Coefficient': best_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=1)
ax.set_xlabel('Coefficient Value')
ax.set_title('Logistic Regression Coefficients\n(Red = increases risk, Green = decreases risk)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))

## 12. Predict on New Patient Data

In [ ]:
def predict_patient(patient_data: dict) -> dict:
    """
    Predict heart disease risk for a new patient.
    
    Parameters:
    - age: int (years)
    - sex: 0=female, 1=male
    - cp: 0=typical angina, 1=atypical, 2=non-anginal, 3=asymptomatic
    - trestbps: resting blood pressure (mmHg)
    - chol: serum cholesterol (mg/dl)
    - fbs: fasting blood sugar >120 mg/dl (0/1)
    - restecg: 0=normal, 1=ST-T abnormal, 2=LV hypertrophy
    - thalach: max heart rate achieved
    - exang: exercise-induced angina (0/1)
    - oldpeak: ST depression induced by exercise
    - slope: 0=upsloping, 1=flat, 2=downsloping
    - ca: number of major vessels (0-3)
    - thal: 0=normal, 1=fixed defect, 2=reversible defect
    """
    features = pd.DataFrame([patient_data])
    features_scaled = scaler.transform(features)
    pred = best_model.predict(features_scaled)[0]
    prob = best_model.predict_proba(features_scaled)[0][1]
    
    if prob < 0.25:   risk = 'LOW'
    elif prob < 0.50: risk = 'MODERATE'
    elif prob < 0.75: risk = 'HIGH'
    else:             risk = 'VERY HIGH'
    
    return {
        'prediction': 'Heart Disease' if pred == 1 else 'No Heart Disease',
        'probability': f'{prob*100:.1f}%',
        'risk_level': risk
    }


# Example 1: High-risk patient
high_risk_patient = {
    'age': 67, 'sex': 1, 'cp': 0, 'trestbps': 160, 'chol': 286,
    'fbs': 0, 'restecg': 2, 'thalach': 108, 'exang': 1,
    'oldpeak': 1.5, 'slope': 1, 'ca': 3, 'thal': 2
}
result1 = predict_patient(high_risk_patient)
print('=== Patient 1 (High Risk) ===')
print(f"  Prediction:  {result1['prediction']}")
print(f"  Probability: {result1['probability']}")
print(f"  Risk Level:  {result1['risk_level']}")

# Example 2: Low-risk patient
low_risk_patient = {
    'age': 45, 'sex': 0, 'cp': 2, 'trestbps': 120, 'chol': 200,
    'fbs': 0, 'restecg': 0, 'thalach': 170, 'exang': 0,
    'oldpeak': 0.0, 'slope': 0, 'ca': 0, 'thal': 0
}
result2 = predict_patient(low_risk_patient)
print('\n=== Patient 2 (Low Risk) ===')
print(f"  Prediction:  {result2['prediction']}")
print(f"  Probability: {result2['probability']}")
print(f"  Risk Level:  {result2['risk_level']}")

## 13. Final Summary

In [ ]:
print('=' * 55)
print('       HEART DISEASE PREDICTION — FINAL RESULTS')
print('=' * 55)
print(f'  Dataset:          Cleveland Heart Disease (UCI)')
print(f'  Total samples:    {len(df)}')
print(f'  Training samples: {len(X_train)}')
print(f'  Test samples:     {len(X_test)}')
print(f'  Features:         {X_final.shape[1]}')
print(f'  Algorithm:        Logistic Regression')
print()
print(f'  Test Accuracy:    {acc_opt*100:.2f}%')
print(f'  Precision:        {precision_opt*100:.2f}%')
print(f'  Recall:           {recall_opt*100:.2f}%')
print(f'  F1 Score:         {f1_opt*100:.2f}%')
print(f'  ROC-AUC:          {roc_auc_opt:.4f}')
print('=' * 55)